# AutoFuse 使能进阶

本节面向项目实践，围绕 GE 和 Inductor 两条编译路线，对比 AutoFuse 的使能入口，介绍常用环境变量及其使用边界，为后续 PyTorch 和 TensorFlow 项目实践提供可复用的配置参考。

本节学习大纲如下：

- 通用术语与相关概念
- 两条路线的使能方式对比
- GE 路线常用环境变量
- Inductor 路线常用环境变量

## 1. 通用术语与相关概念

为便于理解后续内容，学习本课程前请先了解如下术语、缩略语及相关概念。

<table align="left">
  <thead>
    <tr>
      <th>术语</th>
      <th>说明</th>
      <th>在本课程中的理解方式</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>inductor_npu_ext</td>
      <td>PyTorch 侧用于接入 Ascend NPU 编译后端的扩展包</td>
      <td>Inductor 路线使能 AutoFuse 的入口，在调用 torch.compile 前导入即可注册相关编译扩展</td>
    </tr>
    <tr>
      <td>TorchAir</td>
      <td>昇腾开源的 PyTorch 深度融合适配仓库</td>
      <td>Inductor 路线用例的 inductor_npu_ext 扩展从该仓库源码安装</td>
    </tr>
    <tr>
      <td>npu_bridge</td>
      <td>昇腾 TensorFlow Adapter 提供的 Python 接入包</td>
      <td>导入后在框架初始化阶段注册 NPU 相关能力，是 TensorFlow 用例接入 NPU 的入口</td>
    </tr>
    <tr>
      <td>Eager</td>
      <td>PyTorch 的默认执行模式，算子按调用顺序逐个立即执行，不经过图编译</td>
      <td>作为对照基准，用于验证融合路径的数值正确性并对比性能差异</td>
    </tr>
    <tr>
      <td>Profiling</td>
      <td>采集模型执行过程中算子级性能数据的方法</td>
      <td>用于查看实际执行的 Kernel 列表和耗时，是确认融合与性能收益的主要手段</td>
    </tr>
    <tr>
      <td>msprof</td>
      <td>CANN 提供的 Profiling 数据解析与导出工具</td>
      <td>将采集的原始性能数据导出为 op_summary 等可读结果</td>
    </tr>
    <tr>
      <td>Dump 图</td>
      <td>图编译过程中导出的中间计算图文件</td>
      <td>用于直观查看融合前后的图结构变化，辅助确认融合是否发生</td>
    </tr>
  </tbody>
</table>


## 2. 两条路线的使能方式对比

AutoFuse 的使能方式取决于框架进入的编译路线。配置前，应先确认项目的框架入口和编译方式，再选择对应的使能方法。两条路线的使能入口和适用场景如下：

<table align="left">
<thead><tr><th>对接路线</th><th>使能方式</th><th>适用场景</th></tr></thead>
<tbody>
<tr><td>GE 路线</td><td>在图编译前设置环境变量 <code>AUTOFUSE_FLAGS="--enable_autofuse=true"</code></td><td>经 GE 图编译的场景，如 TensorFlow 图模式、ATC 离线编译</td></tr>
<tr><td>Inductor 路线</td><td>在调用 <code>torch.compile</code> 前导入 <code>inductor_npu_ext</code> 扩展</td><td>PyTorch 通过 <code>torch.compile</code> 编译模型的场景</td></tr>
</tbody>
</table>
<div style="clear:left"></div>

两条路线的最简使能方式分别如下。

GE 路线通过环境变量开启 AutoFuse，且应在图编译开始前完成设置：

```bash
export AUTOFUSE_FLAGS="--enable_autofuse=true"
```

Inductor 路线需要先导入扩展，再通过 <code>torch.compile</code> 编译模型：

```python
import inductor_npu_ext  # 注册 Inductor 路线相关扩展

compiled_model = torch.compile(model, dynamic=False, fullgraph=True)
```

其中 <code>torch.compile</code> 的编译模式说明可进一步参考 [TorchNPU 官方文档](https://www.hiascend.com/document/detail/zh/Pytorch/latest/userguide/torchcompile/docs/zh/torch_compile/pytorch_compilation_mode.md)。

两条路线的使能入口不能互相替代：GE 路线的 <code>AUTOFUSE_FLAGS</code> 对 PyTorch 的 Inductor 编译路线不起作用；Inductor 路线也不需要通过该环境变量开启 AutoFuse。无论采用哪条路线，使能都只是必要条件之一，最终是否融合仍取决于融合条件和后端支持范围。

## 3. GE 路线常用环境变量

GE 路线主要通过 <code>AUTOFUSE_FLAGS</code> 配置 AutoFuse 功能，通过 <code>AUTOFUSE_DFX_FLAGS</code> 配置调试信息。以下配置均应在<strong>图编译开始前</strong>设置，并且只对当前 Shell 及其子进程生效。

### 3.1 AUTOFUSE_FLAGS 环境变量控制点

#### --enable_autofuse

控制整体自动融合功能是否开启。

<strong>取值：</strong>

- <code>true</code>：开启 AutoFuse 基础功能。
- <code>false</code>：关闭 AutoFuse；默认值为 <code>false</code>。

如果未配置该参数，整体自动融合功能保持关闭。整体功能关闭时，其他 AutoFuse 控制点也不会生效。

<strong>配置示例：</strong>

```bash
export AUTOFUSE_FLAGS="--enable_autofuse=true"
```

<strong>使用说明：</strong>开启后，满足条件的 Elemwise、Broadcast 类算子可以参与自动融合；是否实际融合仍需通过 Profiling 数据或 Dump 图确认。

#### --autofuse_enable_pass

控制扩展融合能力是否开启。

<strong>取值：</strong>

- <code>reduce</code>：开启 Reduce 融合能力。
- <code>concat</code>：开启 Concat 融合能力。

多个取值使用英文逗号分隔。该参数默认为空，Reduce 类和 Concat 类算子的融合能力目前默认不开启。

<strong>配置示例：</strong>

```bash
export AUTOFUSE_FLAGS="--enable_autofuse=true;--autofuse_enable_pass=reduce,concat"
```

<strong>使用约束：</strong>该参数与 <code>--autofuse_disable_pass</code> 不应同时配置相同的取值。

多个配置项写在<strong>同一个</strong> <code>AUTOFUSE_FLAGS</code> 字符串中，并使用<strong>英文分号</strong>分隔；<code>--autofuse_enable_pass</code> 中的多个融合类型使用<strong>英文逗号</strong>分隔。

### 3.2 AUTOFUSE_DFX_FLAGS 环境变量控制点

#### --codegen_compile_debug

控制是否保留融合算子生成过程中的中间文件。

<strong>取值：</strong>

- <code>true</code>：保留相关文件。
- <code>false</code>：不保留相关文件；默认值为 <code>false</code>。

<strong>配置示例：</strong>

```bash
export AUTOFUSE_DFX_FLAGS="--codegen_compile_debug=true"
```

设置为 <code>true</code> 后，脚本执行目录下可能生成 <code>kernel_meta_*</code> 文件夹，其中包含 Kernel 源码、Tiling 源码、CMake 工程和编译结果；同时，<code>{debug_dir}/autofuse_compile_debug/</code> 路径下会保存融合算子处理过程中的 Dump 图。

<strong>使用约束：</strong>无。

#### --debug_dir

指定融合过程 Dump 图的保存路径。路径支持大小写字母（a-z、A-Z）、数字（0-9）、下划线（_）、中划线（-）、句点（.）和中文字符。执行用户需要对该路径具有读、写和执行权限。未指定时，文件保存至脚本执行时的当前目录。

<strong>配置示例：</strong>

```bash
export AUTOFUSE_DFX_FLAGS="--codegen_compile_debug=true;--debug_dir=/path/to/dump"
```

<strong>使用约束：</strong>需要先配置 <code>--codegen_compile_debug=true</code>，开启自动融合 Debug 开关后该路径配置才有意义。

配置完成后，可以通过以下命令检查当前 Shell 中的变量值：

```bash
echo "$AUTOFUSE_FLAGS"
echo "$AUTOFUSE_DFX_FLAGS"
```

上述命令只能确认变量已经写入环境，不能证明目标算子已经融合。后续 PyTorch 和 TensorFlow 项目实践中，将结合 Dump 图、Profiling 等信息进一步验证融合结果。

## 4. Inductor 路线常用环境变量

Inductor 路线的功能使能入口是导入 <code>inductor_npu_ext</code>，同时支持使用环境变量配置 AutoFuse 调试信息和编译行为。以下配置可能改变执行行为或增加编译开销，建议<strong>仅在开发调测阶段按需使用</strong>，正式性能测试和实际部署前应关闭。

### 4.1 TORCH_COMPILE_DEBUG

作用：PyTorch 原生调试环境变量，用于启用详细调试日志，并保存编译过程中的中间产物。

<strong>使用方法：</strong>

```bash
export TORCH_COMPILE_DEBUG=1
```

启用后，脚本执行目录下会生成 <code>torch_compile_debug</code> 目录，用于保存编译过程中的调试信息和相关中间产物，例如 Kernel 文件。

<strong>注意：</strong>如果多次执行相同脚本时命中缓存，后续执行可能跳过编译，因而看不到新的编译过程。此时可以配合 <code>TORCHINDUCTOR_FORCE_DISABLE_CACHES</code> 使用。

### 4.2 TORCHINDUCTOR_FORCE_DISABLE_CACHES

作用：PyTorch 原生环境变量，用于禁用 Inductor 缓存，使每次执行都重新编译。

<strong>使用方法：</strong>

```bash
export TORCHINDUCTOR_FORCE_DISABLE_CACHES=1
```

<strong>注意：</strong>该环境变量主要用于编译问题定位，会显著增加图启动耗时。正常运行、正式性能测试和实际部署时不建议使用。

### 4.3 AUTOFUSE_DFX_FLAGS 环境变量控制点

Inductor 路线同样支持 <code>AUTOFUSE_DFX_FLAGS</code>，用于配置 AutoFuse 编译调试信息。其配置方式与 GE 路线一致，但不会生成 <code>kernel_meta_*</code> 目录。

<strong>使用方法：</strong>

```bash
export AUTOFUSE_DFX_FLAGS="--codegen_compile_debug=true"
```

其中，<code>--codegen_compile_debug</code> 用于控制是否保留融合算子生成过程中的调试信息，<code>--debug_dir</code> 用于指定 Dump 图的保存路径。其他取值和使用约束请参见 GE 路线中的 <code>AUTOFUSE_DFX_FLAGS</code> 环境变量控制点。

这些调试配置不能替代功能使能入口。例如，只设置相关调试环境变量而未在 <code>torch.compile</code> 前导入 <code>inductor_npu_ext</code>，不会产生预期的 Inductor 路线融合结果或调试产物。

## 课后练习

请根据本节学习内容完成以下题目进行自测。

1. （单选题）GE 路线希望开启 AutoFuse 基础功能，应设置哪个变量？
    A. `TORCH_COMPILE_DEBUG`
    B. `AUTOFUSE_FLAGS`
    C. `TORCHINDUCTOR_FORCE_DISABLE_CACHES`
    D. `AUTOFUSE_DFX_FLAGS`

2. （判断题）只要设置 `AUTOFUSE_DFX_FLAGS="--codegen_compile_debug=true"`，就一定已经打开 GE 路线的 AutoFuse 功能。

3. （判断题）Inductor 路线应在调用 `torch.compile` 前导入 `inductor_npu_ext`。

4. （判断题）开启 AutoFuse 后程序执行无报错，即可确认目标算子已完成融合。

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.02_answer.txt